# 🧠 Deteksi Anomali Ekonomi Makro — Autoencoder

**Early Warning System Krisis Ekonomi menggunakan Autoencoder (Deep Learning)**

---

Notebook ini mengimplementasikan pendekatan **Autoencoder** untuk mendeteksi anomali pada indikator ekonomi makro sebagai sistem peringatan dini (*Early Warning System*) krisis ekonomi.

**Metodologi:**
- Arsitektur: Encoder → Bottleneck → Decoder (kompresi non-linear)
- Threshold: Persentil ke-92 dari *reconstruction error*
- Evaluasi: Precision, Recall, F1-Score terhadap *ground truth* krisis historis
- Interpretasi: Feature importance berbasis *mean reconstruction error difference*

**Dataset:** 14 indikator makroekonomi, 49 negara, periode 1990–2024 (World Bank Open Data)  
**Disusun oleh:** Deka

## 1. Instalasi & Import Library

In [ ]:
# Instalasi dependensi (jalankan sekali jika belum terinstall)
!pip install tensorflow scikit-learn matplotlib seaborn pandas numpy -q

In [ ]:
# ── Import Library ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, LeakyReLU
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Konfigurasi visualisasi
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Load & Eksplorasi Data

In [ ]:
# ── Load dataset ────────────────────────────────────────────────────────
df = pd.read_csv('../data_cleaned.csv')

print(f"Shape dataset: {df.shape}")
print(f"Jumlah negara: {df['economy'].nunique()}")
print(f"Rentang tahun: {df['year'].min()} – {df['year'].max()}")
print(f"\nDistribusi crisis_label:")
print(df['crisis_label'].value_counts())
print(f"\nPersentase krisis: {df['crisis_label'].mean()*100:.2f}%")

df.head()

In [ ]:
# ── Daftar 14 fitur indikator makroekonomi ──────────────────────────────
FEATURE_COLS = [
    'GDP_Growth', 'GDP_PerCapita_Growth', 'Inflation_CPI',
    'Total_Reserves', 'Unemployment', 'Current_Account_GDP',
    'Trade_GDP', 'FDI_Inflows_GDP', 'Exports_GDP', 'Imports_GDP',
    'Gross_Savings_GDP', 'Exchange_Rate', 'Manufacturing_Value',
    'Investment_GDP'
]

# Label deskriptif untuk visualisasi
FEATURE_LABELS = {
    'GDP_Growth': 'Pertumbuhan PDB (%)',
    'GDP_PerCapita_Growth': 'PDB per Kapita Growth (%)',
    'Inflation_CPI': 'Inflasi CPI (%)',
    'Total_Reserves': 'Cadangan Devisa',
    'Unemployment': 'Pengangguran (%)',
    'Current_Account_GDP': 'Neraca Berjalan/PDB (%)',
    'Trade_GDP': 'Perdagangan/PDB (%)',
    'FDI_Inflows_GDP': 'FDI Masuk/PDB (%)',
    'Exports_GDP': 'Ekspor/PDB (%)',
    'Imports_GDP': 'Impor/PDB (%)',
    'Gross_Savings_GDP': 'Tabungan Bruto/PDB (%)',
    'Exchange_Rate': 'Nilai Tukar (LCU/USD)',
    'Manufacturing_Value': 'Manufaktur/PDB (%)',
    'Investment_GDP': 'Investasi/PDB (%)'
}

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"Missing values per fitur:")
print(df[FEATURE_COLS].isnull().sum())
print(f"\nStatistik deskriptif (data sudah ter-standardisasi z-score):")
df[FEATURE_COLS].describe().round(3)

In [ ]:
# ── Visualisasi distribusi fitur ────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    ax.hist(df[col], bins=50, alpha=0.7, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1, label='Mean')
    ax.tick_params(labelsize=8)

# Hapus subplot kosong (ada 15 slot, 14 fitur)
axes[-1].set_visible(False)

fig.suptitle('Distribusi 14 Indikator Makroekonomi (z-score)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df[FEATURE_COLS].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8},
    ax=ax, vmin=-1, vmax=1
)
ax.set_title('Matriks Korelasi Antar Indikator Makroekonomi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Persiapan Data untuk Autoencoder

In [ ]:
# ── Siapkan matriks fitur ───────────────────────────────────────────────
# Data sudah ter-standardisasi (z-score) dari preprocessing sebelumnya
X = df[FEATURE_COLS].values
y_true = df['crisis_label'].values

print(f"Shape fitur (X): {X.shape}")
print(f"Shape label (y): {y_true.shape}")
print(f"Jumlah krisis (label=1): {y_true.sum()} ({y_true.mean()*100:.1f}%)")
print(f"Jumlah normal (label=0): {(y_true==0).sum()} ({(1-y_true.mean())*100:.1f}%)")

In [ ]:
# ── Split data: Train (80%) dan Test (20%) ──────────────────────────────
# Autoencoder dilatih pada SELURUH data (unsupervised) tanpa melihat label
# Split digunakan untuk monitoring training loss vs validation loss

X_train, X_val = train_test_split(X, test_size=0.2, random_state=SEED)

print(f"Training set  : {X_train.shape}")
print(f"Validation set: {X_val.shape}")

## 4. Arsitektur Autoencoder

Autoencoder diracik dengan susunan **Encoder → Bottleneck → Decoder** sebagaimana dirancang pada laporan:
- **Encoder**: Memampatkan 14 dimensi fitur melalui lapisan-lapisan yang semakin menyempit
- **Bottleneck**: Representasi laten terkompres (dimensi rendah)
- **Decoder**: Merekonstruksi data kembali ke 14 dimensi asli

Data perekonomian dengan **reconstruction error** tertinggi (melewati threshold persentil ke-92) diklasifikasikan sebagai **anomali/krisis**.

In [ ]:
# ── Membangun arsitektur Autoencoder ────────────────────────────────────
input_dim = X.shape[1]  # 14 fitur

# === ENCODER ===
input_layer = Input(shape=(input_dim,), name='input')

# Layer 1: 14 → 32
encoded = Dense(32, name='encoder_1')(input_layer)
encoded = BatchNormalization(name='bn_enc_1')(encoded)
encoded = LeakyReLU(alpha=0.1, name='lrelu_enc_1')(encoded)
encoded = Dropout(0.2, name='dropout_enc_1')(encoded)

# Layer 2: 32 → 16
encoded = Dense(16, name='encoder_2')(encoded)
encoded = BatchNormalization(name='bn_enc_2')(encoded)
encoded = LeakyReLU(alpha=0.1, name='lrelu_enc_2')(encoded)
encoded = Dropout(0.2, name='dropout_enc_2')(encoded)

# === BOTTLENECK (representasi laten) ===
bottleneck_dim = 6  # Kompresi: 14 → 6 dimensi
bottleneck = Dense(bottleneck_dim, activation='relu', name='bottleneck')(encoded)

# === DECODER ===
# Layer 3: 6 → 16
decoded = Dense(16, name='decoder_1')(bottleneck)
decoded = BatchNormalization(name='bn_dec_1')(decoded)
decoded = LeakyReLU(alpha=0.1, name='lrelu_dec_1')(decoded)
decoded = Dropout(0.2, name='dropout_dec_1')(decoded)

# Layer 4: 16 → 32
decoded = Dense(32, name='decoder_2')(decoded)
decoded = BatchNormalization(name='bn_dec_2')(decoded)
decoded = LeakyReLU(alpha=0.1, name='lrelu_dec_2')(decoded)
decoded = Dropout(0.2, name='dropout_dec_2')(decoded)

# Output: 32 → 14 (rekonstruksi)
output_layer = Dense(input_dim, activation='linear', name='output')(decoded)

# === BUILD MODEL ===
autoencoder = Model(inputs=input_layer, outputs=output_layer, name='Autoencoder_EWS')

# Kompilasi
autoencoder.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'  # Mean Squared Error sebagai fungsi loss
)

# Ringkasan arsitektur
autoencoder.summary()

In [ ]:
# ── Visualisasi arsitektur ──────────────────────────────────────────────
print("\n" + "="*60)
print("  ARSITEKTUR AUTOENCODER — EARLY WARNING SYSTEM")
print("="*60)
print(f"""  
  Input (14 fitur makroekonomi)
       │
  ┌────▼────┐
  │ Dense 32│ + BatchNorm + LeakyReLU + Dropout(0.2)
  └────┬────┘
       │        ENCODER
  ┌────▼────┐
  │ Dense 16│ + BatchNorm + LeakyReLU + Dropout(0.2)  
  └────┬────┘
       │
  ╔════▼════╗
  ║Dense  {bottleneck_dim} ║  ← BOTTLENECK (representasi laten)
  ╚════╤════╝
       │
  ┌────▼────┐
  │ Dense 16│ + BatchNorm + LeakyReLU + Dropout(0.2)
  └────┬────┘
       │        DECODER
  ┌────▼────┐
  │ Dense 32│ + BatchNorm + LeakyReLU + Dropout(0.2)
  └────┬────┘
       │
  ┌────▼────┐
  │Dense  14│  ← Output (rekonstruksi)
  └─────────┘
  
  Loss Function : Mean Squared Error (MSE)
  Optimizer     : Adam (lr=0.001)
  Bottleneck    : {bottleneck_dim} dimensi (rasio kompresi = {input_dim}/{bottleneck_dim} = {input_dim/bottleneck_dim:.1f}x)
""")

## 5. Training Autoencoder

In [ ]:
# ── Callbacks ───────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
]

# ── Training ────────────────────────────────────────────────────────────
print("🚀 Memulai training Autoencoder...\n")

history = autoencoder.fit(
    X_train, X_train,          # Input = Output (rekonstruksi diri sendiri)
    epochs=200,
    batch_size=32,
    shuffle=True,
    validation_data=(X_val, X_val),
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training selesai pada epoch {len(history.history['loss'])}")
print(f"   Final Train Loss: {history.history['loss'][-1]:.6f}")
print(f"   Final Val Loss  : {history.history['val_loss'][-1]:.6f}")

In [ ]:
# ── Visualisasi Training History ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss curve
axes[0].plot(history.history['loss'], label='Training Loss', color='#2196F3', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='#FF5722', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Training vs Validation Loss', fontweight='bold')
axes[0].legend(frameon=True, fancybox=True, shadow=True)
axes[0].grid(True, alpha=0.3)

# Loss curve (log scale)
axes[1].plot(history.history['loss'], label='Training Loss', color='#2196F3', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='#FF5722', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE (log scale)')
axes[1].set_title('Training Loss (Skala Logaritmik)', fontweight='bold')
axes[1].set_yscale('log')
axes[1].legend(frameon=True, fancybox=True, shadow=True)
axes[1].grid(True, alpha=0.3)

plt.suptitle('📉 Kurva Training Autoencoder', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Deteksi Anomali — Reconstruction Error

Prinsip kerja:
- Autoencoder dilatih untuk merekonstruksi data **normal**
- Data anomali/krisis memiliki pola yang **menyimpang** dari normalitas → rekonstruksi buruk → **error tinggi**
- Threshold ditetapkan pada **persentil ke-92** (~8% contamination rate, sesuai frekuensi historis krisis)

In [ ]:
# ── Hitung Reconstruction Error ─────────────────────────────────────────
# Prediksi (rekonstruksi) seluruh data
X_reconstructed = autoencoder.predict(X, verbose=0)

# Mean Squared Error per observasi
reconstruction_error = np.mean(np.square(X - X_reconstructed), axis=1)

# Simpan ke DataFrame
df['reconstruction_error'] = reconstruction_error

print("📊 Statistik Reconstruction Error:")
print(f"   Mean  : {reconstruction_error.mean():.6f}")
print(f"   Std   : {reconstruction_error.std():.6f}")
print(f"   Min   : {reconstruction_error.min():.6f}")
print(f"   Max   : {reconstruction_error.max():.6f}")
print(f"   Median: {np.median(reconstruction_error):.6f}")

In [ ]:
# ── Tetapkan Threshold (Persentil ke-92) ────────────────────────────────
PERCENTILE_THRESHOLD = 92
threshold = np.percentile(reconstruction_error, PERCENTILE_THRESHOLD)

# Deteksi anomali
df['anomaly_ae'] = (df['reconstruction_error'] > threshold).astype(int)

print(f"🎯 Threshold (persentil ke-{PERCENTILE_THRESHOLD}): {threshold:.6f}")
print(f"\n📋 Hasil Deteksi Anomali:")
print(f"   Normal  (0): {(df['anomaly_ae']==0).sum()} ({(df['anomaly_ae']==0).mean()*100:.1f}%)")
print(f"   Anomali (1): {(df['anomaly_ae']==1).sum()} ({(df['anomaly_ae']==1).mean()*100:.1f}%)")

In [ ]:
# ── Visualisasi Distribusi Reconstruction Error ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Histogram keseluruhan
ax = axes[0]
ax.hist(reconstruction_error[y_true == 0], bins=80, alpha=0.7,
        color='#4CAF50', label='Normal', edgecolor='white')
ax.hist(reconstruction_error[y_true == 1], bins=80, alpha=0.7,
        color='#F44336', label='Krisis (Ground Truth)', edgecolor='white')
ax.axvline(threshold, color='#FF9800', linestyle='--', linewidth=2.5,
           label=f'Threshold P{PERCENTILE_THRESHOLD} = {threshold:.4f}')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi Reconstruction Error', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

# Boxplot per kelas
ax = axes[1]
data_box = [reconstruction_error[y_true == 0], reconstruction_error[y_true == 1]]
bp = ax.boxplot(data_box, labels=['Normal', 'Krisis'],
                patch_artist=True, notch=True, widths=0.5)
bp['boxes'][0].set_facecolor('#4CAF50')
bp['boxes'][1].set_facecolor('#F44336')
for box in bp['boxes']:
    box.set_alpha(0.7)
ax.axhline(threshold, color='#FF9800', linestyle='--', linewidth=2,
           label=f'Threshold = {threshold:.4f}')
ax.set_ylabel('Reconstruction Error (MSE)')
ax.set_title('Reconstruction Error: Normal vs Krisis', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.suptitle('🔍 Analisis Reconstruction Error Autoencoder', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Evaluasi Model terhadap Ground Truth

In [ ]:
# ── Evaluasi: Precision, Recall, F1-Score ───────────────────────────────
y_pred = df['anomaly_ae'].values

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("="*60)
print("  📊 EVALUASI AUTOENCODER vs GROUND TRUTH KRISIS")
print("="*60)
print(f"\n  Precision : {precision:.4f}  — {precision*100:.1f}% anomali terdeteksi benar-benar krisis")
print(f"  Recall    : {recall:.4f}  — {recall*100:.1f}% krisis berhasil terdeteksi")
print(f"  F1-Score  : {f1:.4f}  — Harmonik keseimbangan Precision-Recall")
print(f"\n  Threshold : Persentil ke-{PERCENTILE_THRESHOLD} = {threshold:.6f}")
print("="*60)

# Classification Report lengkap
print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Krisis']))

In [ ]:
# ── Confusion Matrix ────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Anomali'],
    yticklabels=['Normal', 'Krisis'],
    linewidths=2, linecolor='white',
    annot_kws={'size': 16, 'weight': 'bold'},
    ax=ax
)
ax.set_xlabel('Prediksi Autoencoder', fontsize=13)
ax.set_ylabel('Ground Truth', fontsize=13)
ax.set_title(f'Confusion Matrix — Autoencoder\n(Precision={precision:.3f} | Recall={recall:.3f} | F1={f1:.3f})',
             fontsize=14, fontweight='bold')

# Anotasi sel
tn, fp, fn, tp = cm.ravel()
textstr = f'TP={tp}  FP={fp}\nFN={fn}  TN={tn}'
props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.8)
ax.text(2.6, 0.5, textstr, fontsize=11, verticalalignment='top', bbox=props,
        transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curve ───────────────────────────────────────────────────────────
fpr, tpr, thresholds_roc = roc_curve(y_true, reconstruction_error)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color='#2196F3', linewidth=2.5,
        label=f'Autoencoder (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Baseline')
ax.fill_between(fpr, tpr, alpha=0.15, color='#2196F3')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — Autoencoder Anomaly Detection', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=12, frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.show()

print(f"🎯 AUC-ROC Score: {roc_auc:.4f}")

## 8. Analisis Deteksi per Krisis Historis

In [ ]:
# ── Ground Truth: Krisis Historis ───────────────────────────────────────
CRISIS_EVENTS = {
    'Krisis Asia 1997-98': {
        'years': [1997, 1998],
        'countries': ['IDN', 'THA', 'MYS', 'KOR', 'PHL']
    },
    'Krisis Rusia 1998': {
        'years': [1998],
        'countries': ['RUS']
    },
    'Krisis Argentina 2001-02': {
        'years': [2001, 2002],
        'countries': ['ARG']
    },
    'Global Financial Crisis 2008-09': {
        'years': [2008, 2009],
        'countries': df['economy'].unique().tolist()  # Semua negara
    },
    'Krisis Utang Eropa 2010-12': {
        'years': [2010, 2011, 2012],
        'countries': ['GRC', 'PRT', 'IRL', 'ESP', 'ITA']
    },
    'Pandemi COVID-19 2020': {
        'years': [2020],
        'countries': df['economy'].unique().tolist()  # Semua negara
    }
}

print("="*75)
print("  📋 ANALISIS DETEKSI PER KRISIS HISTORIS")
print("="*75)

crisis_results = []
for crisis_name, info in CRISIS_EVENTS.items():
    mask = (df['year'].isin(info['years'])) & (df['economy'].isin(info['countries']))
    subset = df[mask]
    
    if len(subset) == 0:
        continue
    
    detected = subset['anomaly_ae'].sum()
    total = len(subset)
    detection_rate = detected / total * 100
    avg_error = subset['reconstruction_error'].mean()
    
    crisis_results.append({
        'Krisis': crisis_name,
        'Total Observasi': total,
        'Terdeteksi': detected,
        'Detection Rate (%)': round(detection_rate, 1),
        'Avg Recon Error': round(avg_error, 6)
    })
    
    print(f"\n  🔸 {crisis_name}")
    print(f"    Observasi  : {total}")
    print(f"    Terdeteksi : {detected}/{total} ({detection_rate:.1f}%)")
    print(f"    Avg Error  : {avg_error:.6f} (threshold: {threshold:.6f})")
    
    if detected > 0:
        detected_countries = subset[subset['anomaly_ae']==1]['economy'].unique()
        print(f"    Negara     : {', '.join(sorted(detected_countries))}")

print("\n" + "="*75)

# Tabel ringkasan
df_crisis_results = pd.DataFrame(crisis_results)
df_crisis_results

In [ ]:
# ── Visualisasi Detection Rate per Krisis ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#E53935', '#FF7043', '#FFA726', '#42A5F5', '#7E57C2', '#26A69A']
bars = ax.barh(
    df_crisis_results['Krisis'],
    df_crisis_results['Detection Rate (%)'],
    color=colors[:len(df_crisis_results)],
    edgecolor='white', linewidth=1.5, height=0.6
)

# Label nilai di ujung bar
for bar, row in zip(bars, df_crisis_results.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row._4:.1f}% ({row.Terdeteksi}/{row._2})',
            va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Detection Rate (%)', fontsize=12)
ax.set_title('🎯 Tingkat Deteksi Autoencoder per Krisis Historis', fontsize=14, fontweight='bold')
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Reduksi Dimensi & Visualisasi

In [ ]:
# ── PCA — 2D Projection ─────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA: Ground Truth
ax = axes[0]
scatter_normal = ax.scatter(
    X_pca[y_true == 0, 0], X_pca[y_true == 0, 1],
    c='#4CAF50', alpha=0.4, s=15, label='Normal'
)
scatter_crisis = ax.scatter(
    X_pca[y_true == 1, 0], X_pca[y_true == 1, 1],
    c='#F44336', alpha=0.6, s=25, marker='x', label='Krisis (Ground Truth)'
)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — Ground Truth Label', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

# PCA: Prediksi Autoencoder
ax = axes[1]
scatter_normal = ax.scatter(
    X_pca[y_pred == 0, 0], X_pca[y_pred == 0, 1],
    c='#4CAF50', alpha=0.4, s=15, label='Normal'
)
scatter_anomaly = ax.scatter(
    X_pca[y_pred == 1, 0], X_pca[y_pred == 1, 1],
    c='#FF5722', alpha=0.7, s=30, marker='^', label='Anomali (Autoencoder)'
)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — Prediksi Autoencoder', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.suptitle(f'PCA Projection (Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── PCA: Explained Variance Analysis ────────────────────────────────────
pca_full = PCA(random_state=SEED)
pca_full.fit(X)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scree Plot
ax = axes[0]
ax.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
       pca_full.explained_variance_ratio_ * 100,
       color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Komponen Utama')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('Scree Plot', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Cumulative Variance
ax = axes[1]
ax.plot(range(1, len(cumulative_var)+1), cumulative_var,
        'o-', color='#2196F3', linewidth=2, markersize=6)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='90% threshold')
ax.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95% threshold')

# Cari berapa PC untuk 90% dan 95%
n_90 = np.argmax(cumulative_var >= 90) + 1
n_95 = np.argmax(cumulative_var >= 95) + 1
ax.axvline(x=n_90, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)

ax.set_xlabel('Jumlah Komponen Utama')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title(f'Cumulative Variance (90%→{n_90} PC, 95%→{n_95} PC)', fontweight='bold')
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── t-SNE — 2D Projection ───────────────────────────────────────────────
print("⏳ Menjalankan t-SNE (mungkin memakan waktu beberapa menit)...")

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED,
            n_iter=1000, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X)

print("✅ t-SNE selesai!")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# t-SNE: Ground Truth
ax = axes[0]
ax.scatter(X_tsne[y_true == 0, 0], X_tsne[y_true == 0, 1],
           c='#4CAF50', alpha=0.4, s=15, label='Normal')
ax.scatter(X_tsne[y_true == 1, 0], X_tsne[y_true == 1, 1],
           c='#F44336', alpha=0.6, s=25, marker='x', label='Krisis (Ground Truth)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE — Ground Truth Label', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

# t-SNE: Prediksi Autoencoder
ax = axes[1]
ax.scatter(X_tsne[y_pred == 0, 0], X_tsne[y_pred == 0, 1],
           c='#4CAF50', alpha=0.4, s=15, label='Normal')
ax.scatter(X_tsne[y_pred == 1, 0], X_tsne[y_pred == 1, 1],
           c='#FF5722', alpha=0.7, s=30, marker='^', label='Anomali (Autoencoder)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE — Prediksi Autoencoder', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.suptitle('t-SNE 2D Visualization', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── t-SNE gradient berdasarkan Reconstruction Error ─────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=reconstruction_error, cmap='YlOrRd',
    s=12, alpha=0.7, edgecolors='none'
)

cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Reconstruction Error', fontsize=11)

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('t-SNE — Gradien Reconstruction Error', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 10. Feature Importance — Analisis Kontribusi Indikator

Mengidentifikasi variabel ekonomi makro mana yang paling berkontribusi terhadap deteksi anomali, menggunakan pendekatan **Mean Reconstruction Error Difference** per fitur.

In [ ]:
# ── Feature-wise Reconstruction Error ───────────────────────────────────
# Error per fitur (bukan rata-rata keseluruhan)
feature_errors = np.square(X - X_reconstructed)  # shape: (n_samples, 14)

# Mean error per fitur: anomali vs normal
error_anomaly = feature_errors[y_pred == 1].mean(axis=0)
error_normal = feature_errors[y_pred == 0].mean(axis=0)
error_diff = error_anomaly - error_normal  # Selisih (kontribusi anomali)

# Buat DataFrame importance
df_importance = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Error_Anomaly': error_anomaly,
    'Error_Normal': error_normal,
    'Error_Difference': error_diff,
    'Importance_Ratio': error_anomaly / (error_normal + 1e-10)
}).sort_values('Error_Difference', ascending=False)

print("📊 Feature Importance (berdasarkan Reconstruction Error Difference):")
print("="*75)
df_importance.reset_index(drop=True)

In [ ]:
# ── Visualisasi Feature Importance ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart: Error Difference
ax = axes[0]
df_sorted = df_importance.sort_values('Error_Difference', ascending=True)
colors_bar = ['#F44336' if v > 0 else '#4CAF50' for v in df_sorted['Error_Difference']]
ax.barh(df_sorted['Feature'], df_sorted['Error_Difference'],
        color=colors_bar, edgecolor='white', linewidth=0.5, height=0.7)
ax.set_xlabel('Δ Reconstruction Error (Anomali - Normal)')
ax.set_title('Feature Importance\n(Selisih Error Anomali vs Normal)', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

# Bar chart: perbandingan error anomali vs normal
ax = axes[1]
df_sorted2 = df_importance.sort_values('Error_Anomaly', ascending=True)
y_pos = range(len(df_sorted2))
ax.barh([p - 0.2 for p in y_pos], df_sorted2['Error_Anomaly'],
        height=0.4, color='#F44336', alpha=0.8, label='Anomali', edgecolor='white')
ax.barh([p + 0.2 for p in y_pos], df_sorted2['Error_Normal'],
        height=0.4, color='#4CAF50', alpha=0.8, label='Normal', edgecolor='white')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(df_sorted2['Feature'])
ax.set_xlabel('Mean Reconstruction Error')
ax.set_title('Reconstruction Error per Fitur\n(Anomali vs Normal)', fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('🔬 Analisis Feature Importance — Autoencoder', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: Feature Error per Krisis ───────────────────────────────────
crisis_feature_errors = []

for crisis_name, info in CRISIS_EVENTS.items():
    mask = (df['year'].isin(info['years'])) & (df['economy'].isin(info['countries']))
    idx = df[mask].index
    
    if len(idx) > 0:
        mean_errors = feature_errors[idx].mean(axis=0)
        crisis_feature_errors.append(mean_errors)

crisis_names = list(CRISIS_EVENTS.keys())
df_heatmap = pd.DataFrame(
    crisis_feature_errors,
    index=crisis_names,
    columns=FEATURE_COLS
)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    df_heatmap, annot=True, fmt='.3f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Mean Reconstruction Error'}
)
ax.set_title('Reconstruction Error per Fitur per Krisis Historis',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Krisis')
ax.set_xlabel('Indikator Makroekonomi')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Timeline Anomali — Perspektif Temporal

In [ ]:
# ── Rata-rata Reconstruction Error per tahun ────────────────────────────
yearly_stats = df.groupby('year').agg(
    mean_error=('reconstruction_error', 'mean'),
    median_error=('reconstruction_error', 'median'),
    anomaly_count=('anomaly_ae', 'sum'),
    total_count=('anomaly_ae', 'count'),
    crisis_count=('crisis_label', 'sum')
).reset_index()

yearly_stats['anomaly_pct'] = yearly_stats['anomaly_count'] / yearly_stats['total_count'] * 100
yearly_stats['crisis_pct'] = yearly_stats['crisis_count'] / yearly_stats['total_count'] * 100

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

# Panel 1: Reconstruction Error Timeline
ax = axes[0]
ax.plot(yearly_stats['year'], yearly_stats['mean_error'],
        'o-', color='#2196F3', linewidth=2, markersize=5, label='Mean Recon Error')
ax.fill_between(yearly_stats['year'], yearly_stats['mean_error'], alpha=0.15, color='#2196F3')
ax.axhline(threshold, color='#FF9800', linestyle='--', linewidth=2,
           label=f'Threshold (P{PERCENTILE_THRESHOLD})', alpha=0.8)

# Tandai tahun krisis besar
crisis_years_major = [1997, 1998, 2001, 2008, 2009, 2010, 2020]
for cy in crisis_years_major:
    if cy in yearly_stats['year'].values:
        ax.axvspan(cy-0.4, cy+0.4, alpha=0.15, color='red')

ax.set_ylabel('Mean Reconstruction Error', fontsize=11)
ax.set_title('📈 Timeline Reconstruction Error (1990–2024)', fontsize=14, fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

# Panel 2: Persentase deteksi per tahun
ax = axes[1]
width = 0.35
ax.bar(yearly_stats['year'] - width/2, yearly_stats['anomaly_pct'],
       width, color='#FF5722', alpha=0.8, label='Anomali Terdeteksi (%)', edgecolor='white')
ax.bar(yearly_stats['year'] + width/2, yearly_stats['crisis_pct'],
       width, color='#9C27B0', alpha=0.6, label='Ground Truth Krisis (%)', edgecolor='white')

ax.set_xlabel('Tahun', fontsize=11)
ax.set_ylabel('Persentase (%)', fontsize=11)
ax.set_title('Perbandingan: Anomali Terdeteksi vs Ground Truth per Tahun', fontsize=14, fontweight='bold')
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Top Anomali Terdeteksi

In [ ]:
# ── Top 30 Observasi dengan Reconstruction Error Tertinggi ──────────────
top_anomalies = df.nlargest(30, 'reconstruction_error')[[
    'economy', 'year', 'reconstruction_error', 'anomaly_ae', 'crisis_label'
]].copy()

top_anomalies['Status'] = top_anomalies.apply(
    lambda row: '✅ True Positive' if row['anomaly_ae']==1 and row['crisis_label']==1
    else ('⚠️ False Positive' if row['anomaly_ae']==1 and row['crisis_label']==0
    else '❌ Missed'), axis=1
)

print("🔝 Top 30 Observasi dengan Reconstruction Error Tertinggi:")
print("="*75)
top_anomalies.reset_index(drop=True)

In [ ]:
# ── Negara dengan anomali terbanyak ─────────────────────────────────────
country_anomalies = df[df['anomaly_ae']==1].groupby('economy').agg(
    anomaly_count=('anomaly_ae', 'sum'),
    avg_error=('reconstruction_error', 'mean'),
    years_detected=('year', lambda x: ', '.join(map(str, sorted(x))))
).sort_values('anomaly_count', ascending=False)

print("🌍 Negara dengan Anomali Terbanyak:")
print("="*75)
country_anomalies.head(15)

In [ ]:
# ── Visualisasi Top-15 negara ───────────────────────────────────────────
top15_countries = country_anomalies.head(15)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    top15_countries.index[::-1],
    top15_countries['anomaly_count'][::-1],
    color=plt.cm.Reds(np.linspace(0.3, 0.9, 15)),
    edgecolor='white', linewidth=1
)

for bar, count in zip(bars, top15_countries['anomaly_count'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Jumlah Anomali Terdeteksi', fontsize=12)
ax.set_title('🌍 Top 15 Negara dengan Anomali Terbanyak (Autoencoder)',
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Analisis Bottleneck (Representasi Laten)

In [ ]:
# ── Ekstraksi representasi bottleneck ────────────────────────────────────
# Buat model encoder terpisah (input → bottleneck)
encoder_model = Model(
    inputs=autoencoder.input,
    outputs=autoencoder.get_layer('bottleneck').output,
    name='Encoder'
)

# Dapatkan representasi laten
latent_representation = encoder_model.predict(X, verbose=0)

print(f"Shape representasi laten: {latent_representation.shape}")
print(f"Dimensi bottleneck: {latent_representation.shape[1]}")

# Visualisasi 2D dari bottleneck (ambil 2 dimensi pertama)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(latent_representation[y_true==0, 0], latent_representation[y_true==0, 1],
           c='#4CAF50', alpha=0.4, s=12, label='Normal')
ax.scatter(latent_representation[y_true==1, 0], latent_representation[y_true==1, 1],
           c='#F44336', alpha=0.6, s=20, marker='x', label='Krisis')
ax.set_xlabel('Latent Dim 1')
ax.set_ylabel('Latent Dim 2')
ax.set_title('Bottleneck Space — Ground Truth', fontweight='bold')
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)

ax = axes[1]
scatter = ax.scatter(
    latent_representation[:, 0], latent_representation[:, 1],
    c=reconstruction_error, cmap='YlOrRd', alpha=0.7, s=12
)
plt.colorbar(scatter, ax=ax, label='Reconstruction Error')
ax.set_xlabel('Latent Dim 1')
ax.set_ylabel('Latent Dim 2')
ax.set_title('Bottleneck Space — Reconstruction Error', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('🔬 Representasi Laten Bottleneck (6D → 2D slice)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. Ekspor Hasil

In [ ]:
# ── Simpan hasil deteksi ────────────────────────────────────────────────
output_cols = ['economy', 'year'] + FEATURE_COLS + [
    'crisis_label', 'reconstruction_error', 'anomaly_ae'
]

df_output = df[output_cols].copy()
df_output.to_csv('../deka/hasil_autoencoder.csv', index=False)

print("✅ Hasil deteksi disimpan ke: deka/hasil_autoencoder.csv")
print(f"   Total baris  : {len(df_output)}")
print(f"   Anomali      : {df_output['anomaly_ae'].sum()}")
print(f"   Kolom output : {list(df_output.columns)}")

## 15. Ringkasan Hasil

In [ ]:
# ── Ringkasan Akhir ─────────────────────────────────────────────────────
print("\n" + "="*70)
print("  🧠 RINGKASAN HASIL — AUTOENCODER ANOMALY DETECTION")
print("  Early Warning System Krisis Ekonomi")
print("="*70)

print(f"""
  ┌─────────────────────────────────────────────────────────────┐
  │  DATASET                                                    │
  │  Observasi    : {len(df):,} (49 negara × 35 tahun)             │
  │  Fitur        : {len(FEATURE_COLS)} indikator makroekonomi              │
  │  Ground Truth : {y_true.sum()} krisis ({y_true.mean()*100:.1f}%)                        │
  ├─────────────────────────────────────────────────────────────┤
  │  ARSITEKTUR AUTOENCODER                                     │
  │  Encoder      : 14 → 32 → 16 → {bottleneck_dim} (bottleneck)            │
  │  Decoder      : {bottleneck_dim} → 16 → 32 → 14 (output)                │
  │  Kompresi     : {input_dim/bottleneck_dim:.1f}x ({input_dim}D → {bottleneck_dim}D)                           │
  │  Loss         : MSE, Optimizer: Adam                        │
  │  Regularisasi : BatchNorm + Dropout(0.2) + EarlyStopping    │
  ├─────────────────────────────────────────────────────────────┤
  │  HASIL DETEKSI                                              │
  │  Threshold    : Persentil ke-{PERCENTILE_THRESHOLD} = {threshold:.6f}            │
  │  Anomali      : {(y_pred==1).sum()} terdeteksi ({(y_pred==1).mean()*100:.1f}%)                     │
  ├─────────────────────────────────────────────────────────────┤
  │  EVALUASI vs GROUND TRUTH                                   │
  │  Precision    : {precision:.4f}                                       │
  │  Recall       : {recall:.4f}                                       │
  │  F1-Score     : {f1:.4f}                                       │
  │  AUC-ROC      : {roc_auc:.4f}                                       │
  └─────────────────────────────────────────────────────────────┘
""")

# Top-3 fitur paling berkontribusi
top3 = df_importance.head(3)
print("  📌 Top 3 Fitur Paling Sensitif terhadap Anomali:")
for i, row in enumerate(top3.itertuples(), 1):
    print(f"     {i}. {row.Feature} (Δ Error = {row.Error_Difference:.4f})")

print("\n" + "="*70)